# Brazil — SARIMAX


## 1. Imports e configurações

In [ ]:
from pathlib import Path
import json, warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")
CSV_PATH=Path("datasets/brazil.csv")
TARGET="goals_brazil"
TEST_SIZE=0.20
RANDOM_STATE=42

from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
SEASONAL_PERIODS=[4,8,12]
MAX_CANDIDATES=25

## 2. Limpeza

In [ ]:
def load_and_clean(path):
    df=pd.read_csv(path).copy()
    df["date"]=pd.to_datetime(df["Date"],format="%d %b %Y",errors="coerce")
    df=df.sort_values("date",kind="stable").drop_duplicates("date",keep="first").reset_index(drop=True)
    score=df["Score"].astype("string").str.extract(r"^\s*(\d+)\s*-\s*(\d+)")
    df["home_goals"]=pd.to_numeric(score[0],errors="coerce")
    df["away_goals"]=pd.to_numeric(score[1],errors="coerce")
    df["is_home"]=df["Match"].str.startswith("Brazil v ").astype(int)
    df["opponent"]=np.where(df["is_home"].eq(1),df["Match"].str.replace(r"^Brazil v ","",regex=True),df["Match"].str.replace(r" v Brazil$","",regex=True))
    df["goals_brazil"]=np.where(df["is_home"].eq(1),df["home_goals"],df["away_goals"])
    df["goals_against"]=np.where(df["is_home"].eq(1),df["away_goals"],df["home_goals"])
    df["points"]=df["Result"].map({"W":3,"D":1,"L":0})
    df=df.dropna(subset=["date","Competition","opponent","goals_brazil","goals_against","points"]).copy()
    for c in ["goals_brazil","goals_against","points"]: df[c]=df[c].astype(float)
    return df.reset_index(drop=True)
raw=load_and_clean(CSV_PATH)
print("Linhas após limpeza:",len(raw)); display(raw.head())


## 3. Exógenas e divisão temporal

In [ ]:
raw["days_since_last_match"]=raw["date"].diff().dt.days.clip(lower=0)
raw["month"]=raw["date"].dt.month; raw["month_sin"]=np.sin(2*np.pi*raw["month"]/12); raw["month_cos"]=np.cos(2*np.pi*raw["month"]/12)
raw["day_of_year"]=raw["date"].dt.dayofyear; raw["doy_sin"]=np.sin(2*np.pi*raw["day_of_year"]/365.25); raw["doy_cos"]=np.cos(2*np.pi*raw["day_of_year"]/365.25); raw["year"]=raw["date"].dt.year
num=["is_home","days_since_last_match","month_sin","month_cos","doy_sin","doy_cos","year"]; cat=["Competition"]; cols=num+cat
data=raw.dropna(subset=[TARGET]+cols).reset_index(drop=True); split_idx=int(len(data)*(1-TEST_SIZE)); train=data.iloc[:split_idx].copy(); test=data.iloc[split_idx:].copy()
pre=ColumnTransformer([("numeric",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num),("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat)])
X_train=pre.fit_transform(train[cols]); X_test=pre.transform(test[cols]); y_train=train[TARGET].to_numpy(float); y_test=test[TARGET].to_numpy(float)
print("Treino:",len(train),"Teste:",len(test))

## 4. Busca de `(p,d,q)` e `(P,D,Q,m)`

In [ ]:
def candidates():
    c=[]
    for p,d,q,P,D,Q,m in product([0,1,2],[0,1],[0,1,2],[0,1],[0,1],[0,1],SEASONAL_PERIODS):
        if p==d==q==P==D==Q==0: continue
        c.append(((p,d,q),(P,D,Q,m)))
    rng=np.random.default_rng(RANDOM_STATE); rng.shuffle(c); return c[:MAX_CANDIDATES]
rows=[]; best_bic=np.inf; best_order=None; best_seasonal=None
for order,seasonal in candidates():
    try:
        fit=SARIMAX(y_train,exog=X_train,order=order,seasonal_order=seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
        rows.append({"order":order,"seasonal_order":seasonal,"AIC":fit.aic,"BIC":fit.bic})
        if np.isfinite(fit.bic) and fit.bic<best_bic: best_bic,best_order,best_seasonal=fit.bic,order,seasonal
    except Exception: pass
search_table=pd.DataFrame(rows).sort_values("BIC").reset_index(drop=True)
if best_order is None: raise RuntimeError("Nenhum SARIMAX convergiu")
print("Melhor order:",best_order,"Melhor seasonal:",best_seasonal,"BIC:",round(best_bic,2)); display(search_table.head(10))

## 5. Walk-forward final

In [ ]:
preds=[]
for i in range(len(y_test)):
    yh=np.concatenate([y_train,y_test[:i]]); Xh=np.vstack([X_train,X_test[:i]])
    fit=SARIMAX(yh,exog=Xh,order=best_order,seasonal_order=best_seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
    preds.append(float(np.ravel(fit.get_forecast(steps=1,exog=X_test[i:i+1]).predicted_mean)[0]))
preds=np.asarray(preds); results=pd.DataFrame({"date":test["date"].values,"real":y_test,"prediction":preds,"absolute_error":np.abs(y_test-preds)}); mae=mean_absolute_error(y_test,preds)
print("MAE final walk-forward:",round(mae,4)); display(results.head(10))

## 6. Treino final e salvamento

In [ ]:
OUTPUT_DIR=Path("outputs_sarimax_brazil"); OUTPUT_DIR.mkdir(exist_ok=True)
X_all=pre.fit_transform(data[cols]); y_all=data[TARGET].to_numpy(float)
final=SARIMAX(y_all,exog=X_all,order=best_order,seasonal_order=best_seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
final.save(OUTPUT_DIR/"sarimax_brazil.pkl"); joblib.dump(pre,OUTPUT_DIR/"exog_preprocessor.joblib"); results.to_csv(OUTPUT_DIR/"walk_forward_predictions.csv",index=False)
with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f: json.dump({"order":list(best_order),"seasonal_order":list(best_seasonal),"BIC":float(best_bic),"MAE_walk_forward":float(mae)},f,ensure_ascii=False,indent=2)
print("Salvo em:",OUTPUT_DIR.resolve())